# Parity check: does `coralid` reproduce the legacy notebooks?

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CoralsINPA/RedSeaCoralIdentifier/blob/claude/awesome-archimedes-kvh4c7/notebooks/parity_check.ipynb)

Run this in Colab on a **GPU runtime** (Runtime > Change runtime type > T4). It compares the refactored package with the legacy notebooks and with the numbers recorded earlier. Nothing here changes behavior: it only measures.

**Safe to run and re-run.** The notebook reads the shared Drive and never overwrites a file there. The only things it creates are the folders `<run>/manifest/` (Check C) and `<run>/reports/`, if they do not exist yet, and the files in them: the manifest, and one new summary `parity_<date>.json` (a re-run on the same day gets `_2`, `_3`, ...).

**Photos are checked by orientation.** 221 of the 1,714 photos are portrait (1920x2560 frames). The legacy code squashed them to 2560x1920 without moving the boxes, so its crops on them were wrong and `coralid` is not expected to match it there. Landscape photos must match exactly; portrait photos are reported as information.

| Step | What it does | Time | Reads | Writes |
|---|---|---|---|---|
| Setup | mounts Drive, clones this branch, installs `coralid[clip]` | 3-5 min | GitHub | nothing on Drive |
| Check A | area / aspect / containment filters over all mask JSONs (44,211 masks, expect 16,197 left; 1,493 landscape and 221 portrait frames) | a few min | `<run>/json/*.json` | nothing |
| Check B | zero-shot labels for the 50-image sample: legacy code vs `coralid`, landscape and portrait photos apart | about 15 min on a GPU | `annotation_counts_area25000_cont0.8_ar3.0.csv`, 50 JSONs and photos, the model from Hugging Face, `coral_weak_labels_finetuned-bioclip_taxonomy_masked_best.csv` | nothing |
| Check D | the predictions stored in `assaf_expert_test_set.csv` against its expert labels, cell by cell | seconds | `<run>/assaf_expert_test_set.csv` | nothing |
| Check C | image manifest over all 1,714 photos: every mask is in it and no photo failed | seconds if `manifest.parquet` is saved (it is), else about 40 min to 2 h, **resumable** | all JSONs and photos | `<run>/manifest/` (new folder) if it is not there yet |
| Summary | one PASS / FAIL table | seconds | the results above | `<run>/reports/parity_<date>.json` (new file) |

Run the cells in order. Check C is last on purpose: A, B and D tell you the most and take 20 minutes. If Colab disconnects during Check C, reconnect, run **Setup** and **Config** again, then run Check C again. It continues where it stopped.

**What to paste back to the next session:** the output of the last cell (the JSON block under "PASTE THIS BACK"). A partial run is fine: the checks you skipped show as `NOT RUN`.

Pass criteria are in `src/coralid/parity.py` (A, B), `parity_manifest.py` (C) and `parity_expert.py` (D). Check B needs, on landscape photos, identical key sets, top-1 agreement of at least 99.5% and a maximum absolute probability difference of at most 1e-3. Check C needs all 44,211 masks in the manifest and no unresolved failure; the kept counts are reported as information.

In [ ]:
# ================================ SETTINGS: the only place to edit ================================
# The Drive folder "CoralsMasking" (the root of the shared data):
DATA_ROOT = "/content/drive/MyDrive/colab notebooks/CoralsMasking"
BRANCH = "claude/awesome-archimedes-kvh4c7"
REPO_URL = "https://github.com/CoralsINPA/RedSeaCoralIdentifier"
REPO_DIR = "/content/RedSeaCoralIdentifier"
# ==================================================================================================

import glob
import importlib
import importlib.util
import os
import site
import subprocess
import sys
from importlib import metadata

from google.colab import drive

drive.mount("/content/drive")
os.environ["CORALID_DATA_ROOT"] = DATA_ROOT  # coralid reads the data location from here
if not os.path.isdir(DATA_ROOT):
    raise FileNotFoundError(
        f"DATA_ROOT not found: {DATA_ROOT}\n"
        "Edit DATA_ROOT above. If the folder was shared with you, add a shortcut to it in My Drive first."
    )
if not os.path.isdir(os.path.join(DATA_ROOT, "results_all")):
    raise FileNotFoundError(
        f"DATA_ROOT exists but has no results_all/ folder: {DATA_ROOT}\n"
        "So this is not the shared data folder. Most likely Drive is mounted with a different "
        "Google account than the one the data is shared with: Runtime > Disconnect and delete "
        "runtime, run this cell again and choose that account. Otherwise fix DATA_ROOT."
    )

WATCH = [
    "numpy",
    "pandas",
    "pyarrow",
    "Pillow",
    "opencv-python",
    "opencv-python-headless",
    "torch",
    "torchvision",
    "open_clip_torch",
    "transformers",
]


def installed_versions():
    out = {}
    for name in WATCH:
        try:
            out[name] = metadata.version(name)
        except metadata.PackageNotFoundError:
            out[name] = None
    return out


def run(cmd):
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)


before = installed_versions()

if os.path.isdir(os.path.join(REPO_DIR, ".git")):  # already cloned: update it
    run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH])
    run(["git", "-C", REPO_DIR, "checkout", BRANCH])
    run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH])
    run(["git", "-C", REPO_DIR, "submodule", "update", "--init"])
else:
    run(["git", "clone", "--recurse-submodules", "--branch", BRANCH, REPO_URL, REPO_DIR])
run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}[clip]"])

# An editable install adds a .pth file, and a running interpreter reads those only at startup:
# without this, `import coralid` fails in the session that just installed it.
for site_dir in getattr(site, "getsitepackages", lambda: [])():
    for pth in glob.glob(os.path.join(site_dir, "__editable__*coralid*.pth")):
        site.addpackage(site_dir, os.path.basename(pth), None)
importlib.invalidate_caches()
if importlib.util.find_spec("coralid") is None:  # last resort: use the clone's src folder directly
    sys.path.insert(0, f"{REPO_DIR}/src")

after = installed_versions()
replaced = {
    k: (before[k], after[k]) for k in WATCH if before[k] is not None and before[k] != after[k]
}

print("\nInstalled versions (pip metadata):")
for name in ("opencv-python", "opencv-python-headless", "torch", "open_clip_torch"):
    print(f"  {name:24} {after[name]}")
if (
    after["opencv-python"]
    and after["opencv-python-headless"]
    and after["opencv-python"] != after["opencv-python-headless"]
):
    print(
        "  WARNING: opencv-python and opencv-python-headless differ in version; they share one cv2 folder."
    )

print("\nLoaded modules:")
import cv2
import torch

print("  cv2", cv2.__version__)
print("  torch", torch.__version__)
try:
    import open_clip

    print("  open_clip", open_clip.__version__)
except ImportError as err:
    print("  open_clip: import failed:", err)
print(
    "  GPU:",
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available()
    else "NONE (Check B will be very slow)",
)

import coralid

commit = subprocess.run(
    ["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"], capture_output=True, text=True
).stdout.strip()
print(f"\ncoralid {coralid.__version__} at commit {commit} (branch {BRANCH})")

if replaced:
    raise RuntimeError(
        "pip replaced packages that this Python session may already have loaded: "
        + ", ".join(f"{k} {a} -> {b}" for k, (a, b) in replaced.items())
        + "\n\n>>> RESTART THE RUNTIME now (Runtime > Restart session), then run this cell again and continue. <<<"
    )
print("\nSetup done. No restart needed.")

In [ ]:
import json
import logging
import time
from pathlib import Path

import numpy as np
import pandas as pd

from coralid.config import load_config

logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")
logging.getLogger("coralid").setLevel(logging.INFO)

cfg = load_config(f"{REPO_DIR}/configs/iou60_sta55_pts32.yaml")  # reads CORALID_DATA_ROOT
print("run name:", cfg.run_name)
for label, path in {
    "photos": cfg.images_dir,
    "run folder": cfg.run_dir,
    "mask JSONs": cfg.json_dir,
    "expert set": cfg.run_dir / "assaf_expert_test_set.csv",
    "counts file": cfg.run_dir / "annotation_counts_area25000_cont0.8_ar3.0.csv",
}.items():
    print(f"  {label:11} {'ok     ' if path.exists() else 'MISSING'} {path}")
if not (cfg.images_dir.is_dir() and cfg.json_dir.is_dir()):
    raise FileNotFoundError(
        "photos or mask JSONs not found under DATA_ROOT; check DATA_ROOT in the setup cell"
    )

from coralid.parity_report import format_summary, format_table, paste_block, summarize, write_report

ROWS = {}  # check letter -> its result rows; a re-run of a check replaces its rows

## Check A: geometric filters on all mask JSONs (a few minutes)

Audits the JSON folder (frame sizes, masks, JSONs with a `note`, which are failed images saved as empty), then applies the area, aspect and containment filters to every mask with the RLE-based containment and times them. Expect 1,714 files (1,493 landscape frames, 221 portrait), 44,211 masks and 16,197 masks left. Reads only the JSONs.

In [ ]:
from coralid.manifest import geometric_manifest
from coralid.masks import audit_json_dir
from coralid.parity import check_geometric, check_json_audit

audit = audit_json_dir(cfg.json_dir, progress=True)
print(f"{audit.n_files} JSON files, {audit.n_masks} masks")
print(
    "frame sizes (width x height: files):",
    {f"{w}x{h}": n for (w, h), n in audit.frame_sizes.items()},
)
print("JSONs with a note (images whose segmentation failed):", audit.noted)
print("unreadable JSONs:", audit.unreadable)
print("JSON name differs from image name:", audit.name_mismatches)

timings = {}
geometric = geometric_manifest(cfg.json_dir, cfg.filters, progress=True, timings=timings)
print(
    f"\nreading the JSONs took {timings['load_s']:.0f} s, the filters (RLE containment) {timings['filter_s']:.0f} s"
)
print("masks removed per reason:", geometric["stage1_reason"].value_counts().to_dict())

ROWS["A"] = check_json_audit(audit) + check_geometric(geometric, timings)
print()
print(format_table(ROWS["A"]))

## Check B: zero-shot parity on the 50-image sample (about 15 minutes, GPU)

The next cell is the **legacy reference**: the code of `weak_classification.ipynb` (cells 8 and 14), copied verbatim, set to fine-tuned BioCLIP / taxonomy prompts / masked crops / the "best" sample (the 50 images of the cached counts file, in file order). Its adaptations are listed at the top of the cell. It produces the reference predictions in memory and writes nothing.

The cell after it runs the same configuration through `coralid` and compares the two **separately for landscape and portrait photos**.
- **Landscape photos must match exactly:** identical `(image_name, coral_id)` keys, top-1 agreement of at least 99.5%, maximum absolute probability difference of at most 1e-3. The first run gave 773 of 773 crops, 100% top-1, difference 9e-6.
- **Portrait photos are information only.** The legacy code cut the wrong region there (it kept 69 of the 278 correct crops), so a difference is expected and is the bug that `coralid` fixes.

It also reproduces the legacy counts (2,337 masks, 1,054 after filtering, 842 crops; `coralid` keeps more crops because of the portrait photos) and, for information only, compares with `coral_weak_labels_finetuned-bioclip_taxonomy_masked_best.csv` on Drive (rewritten on 2026-10-11, so a mismatch there is not a failure).

In [ ]:
# ============================================================================================
# LEGACY REFERENCE. The only cell in this notebook that may contain legacy code.
#
# Verbatim copy of notebooks/legacy/weak_classification.ipynb
#   - cell 8  (shared filter parameters & functions), settings left exactly as they are there:
#             finetuned-bioclip / taxonomy / masked / sample = "best", 50 images
#   - cell 14 (weak labeling: model, prompts, crop functions, processing loop)
# Everything not marked ">>> ADAPTED" is byte-for-byte the legacy text. The adaptations are:
#   (1) paths: data_path, run_dir and json_dir are set from the coralid config, because the
#       legacy notebook defined them in earlier cells.
#   (2) sample selection: df_counts is read from the cached counts file the way WC cell 12 does
#       when the file exists (the sample is then df_counts.head(50) in file order, by the legacy code).
#   (3) output: final_df stays in memory instead of being written to output_csv. The "Output:"
#       line printed further down is the legacy path; nothing is written there.
#   (4) legacy section 10 (confidence histogram) is omitted: it saved a PNG into the run folder,
#       over an existing file.
# The legacy loop reads Drive and writes nothing.
# ============================================================================================
# >>> ADAPTED (1): data_path, run_dir and json_dir, which earlier cells of the legacy notebook define.
# >>> They point at the same Drive folders; cfg comes from configs/iou60_sta55_pts32.yaml.
data_path = str(cfg.images_dir)
run_dir = str(cfg.run_dir)
json_dir = str(cfg.json_dir)

# ======================================
# CELL: Shared filter parameters & functions
# (must run before counts cache AND before labeling)
# ======================================
import pycocotools.mask as mask_util
import numpy as np
import cv2
import os

# ======================================
# CONTROLS
# ======================================
SAMPLE_ONLY        = True
SAMPLE_SIZE        = 50
RESUME             = False

MIN_AREA           = 25000
CONTAINMENT_THRESH = 0.80
MAX_ASPECT_RATIO   = 3.0
SHARPNESS_THRESHOLD = 80

# "finetuned-bioclip"  — ReefNet fine-tune of BioCLIP v1 (ViT-B/16, coral-specific)
# "bioclip2"           — BioCLIP v2, trained on 200M biology images (ViT-H/14, stronger general bio)
# "siglip2"            — Google SigLIP 2 SO400M (0.9B, sigmoid loss, better calibrated, 384px)
#                        Use PROMPT_FORMAT="descriptive" — taxonomy strings not useful here
MODEL_VERSION      = "finetuned-bioclip"

# "taxonomy"   — full Linnaean string (best for BioCLIP, matches training format)
# "descriptive"— morphological natural language descriptions
# "ensemble"   — averages taxonomy + descriptive + common-name embeddings (recommended)
# hybrid
PROMPT_FORMAT      = "taxonomy"

SAMPLE_MODE        = "best"
RANDOM_SEED        = 42

USE_MASK           = True
MASK_BACKGROUND    = "black"
MASK_PADDING_PCT   = 0.05
# ======================================

def decode_rle(rle):
    if isinstance(rle['counts'], str):
        rle = {'size': rle['size'], 'counts': rle['counts'].encode('utf-8')}
    return mask_util.decode(rle).astype(bool)

def bbox_intersection(b1, b2):
    x1, y1, w1, h1 = b1
    x2, y2, w2, h2 = b2
    ix = max(0, min(x1+w1, x2+w2) - max(x1, x2))
    iy = max(0, min(y1+h1, y2+h2) - max(y1, y2))
    return ix * iy

def aspect_ratio(bbox):
    _, _, w, h = bbox
    return max(w, h) / max(min(w, h), 1)

def filter_annotations(annotations, min_area, containment_thresh, max_aspect_ratio):
    candidates = [a for a in annotations if a['area'] >= min_area]
    candidates = [a for a in candidates if aspect_ratio(a['bbox']) <= max_aspect_ratio]
    if len(candidates) <= 1:
        return candidates
    to_remove = set()
    for i in range(len(candidates)):
        for j in range(len(candidates)):
            if i == j or candidates[j]['id'] in to_remove:
                continue
            a_small, a_large = candidates[i], candidates[j]
            if a_large['area'] <= a_small['area']:
                continue
            if bbox_intersection(a_small['bbox'], a_large['bbox']) == 0:
                continue
            mask_small = decode_rle(a_small['segmentation'])
            mask_large = decode_rle(a_large['segmentation'])
            containment = np.logical_and(mask_small, mask_large).sum() / a_small['area']
            if containment >= containment_thresh:
                to_remove.add(a_large['id'])
    return [a for a in candidates if a['id'] not in to_remove]


json_files_all = sorted([f for f in os.listdir(json_dir) if f.endswith('.json')])
print(f"✅ Filter functions ready | {len(json_files_all)} JSON files found")

# >>> ADAPTED (2): df_counts, which WC cell 12 builds. Same lines as its "cache file exists" branch;
# >>> the compute-and-write branch is left out (it would write a new cache file to Drive).
import pandas as pd
counts_cache = os.path.join(
    run_dir,
    f'annotation_counts_area{MIN_AREA}_cont{CONTAINMENT_THRESH}_ar{MAX_ASPECT_RATIO}.csv'
)
df_counts = pd.read_csv(counts_cache)
print(f"Loaded cached counts ({len(df_counts)} files) from {counts_cache}")
print(df_counts.head(3))

# ======================================
# CELL 4: Weak Labeling with BioCLIP / BioCLIP-2 (ReefNet)
# ======================================
import os
import torch
import cv2
import json
import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image
import open_clip
import pycocotools.mask as mask_util
import matplotlib.pyplot as plt
import random


# ── 1. MODEL SETUP ──────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

MODEL_IDS = {
    "finetuned-bioclip": ("open_clip",    "hf-hub:ReefNet/finetuned-bioclip"),
    "bioclip2":          ("open_clip",    "hf-hub:imageomics/bioclip-2"),
    "siglip2":           ("transformers", "google/siglip2-so400m-patch14-384"),
}
if MODEL_VERSION not in MODEL_IDS:
    raise ValueError(f"Unknown MODEL_VERSION: {MODEL_VERSION}. Choose from {list(MODEL_IDS)}")

model_backend, model_id = MODEL_IDS[MODEL_VERSION]
print(f"Loading model: {model_id}  (backend: {model_backend})")

if model_backend == "open_clip":
    model, _, preprocess_val = open_clip.create_model_and_transforms(model_id)
    tokenizer = open_clip.get_tokenizer(model_id)
    siglip_processor = None
elif model_backend == "transformers":
    from transformers import AutoProcessor, AutoModel
    siglip_processor = AutoProcessor.from_pretrained(model_id)
    model = AutoModel.from_pretrained(model_id)
    tokenizer = None
    preprocess_val = None

model = model.to(device)
model.eval()
print(f"✅ Model loaded: {MODEL_VERSION}")

# ── 2. GENERA + THREE PROMPT STYLES ─────────────────────────────────────────
# Style A — full Linnaean taxonomy (primary training format for BioCLIP)
genera_taxonomy = {
    "Platygyra":    "Animalia Cnidaria Anthozoa Scleractinia Merulinidae Platygyra",
    "Acropora":     "Animalia Cnidaria Anthozoa Scleractinia Acroporidae Acropora",
    "Stylophora":   "Animalia Cnidaria Anthozoa Scleractinia Pocilloporidae Stylophora",
    "Millepora":    "Animalia Cnidaria Hydrozoa Anthoathecata Milleporidae Millepora",
    "Favites":      "Animalia Cnidaria Anthozoa Scleractinia Faviidae Favites",
    "Dipsastraea":  "Animalia Cnidaria Anthozoa Scleractinia Merulinidae Dipsastraea",
    "Favia":        "Animalia Cnidaria Anthozoa Scleractinia Faviidae Favia",
    "Echinopora":   "Animalia Cnidaria Anthozoa Scleractinia Faviidae Echinopora",
    "Hydnophora":   "Animalia Cnidaria Anthozoa Scleractinia Merulinidae Hydnophora",
    "Porites":      "Animalia Cnidaria Anthozoa Scleractinia Poritidae Porites",
    "Acanthastrea": "Animalia Cnidaria Anthozoa Scleractinia Lobophylliidae Acanthastrea",
    "Cyphastrea":   "Animalia Cnidaria Anthozoa Scleractinia Faviidae Cyphastrea",
    "Goniastrea":   "Animalia Cnidaria Anthozoa Scleractinia Faviidae Goniastrea",
    "Seriatopora":  "Animalia Cnidaria Anthozoa Scleractinia Pocilloporidae Seriatopora",
    "Pocillopora":  "Animalia Cnidaria Anthozoa Scleractinia Pocilloporidae Pocillopora",
    "Montipora":    "Animalia Cnidaria Anthozoa Scleractinia Acroporidae Montipora",
    "Leptastrea":   "Animalia Cnidaria Anthozoa Scleractinia Faviidae Leptastrea",
    "Galaxea":      "Animalia Cnidaria Anthozoa Scleractinia Euphylliidae Galaxea",
    "Lobophyllia":  "Animalia Cnidaria Anthozoa Scleractinia Lobophylliidae Lobophyllia",
    "Pavona":       "Animalia Cnidaria Anthozoa Scleractinia Agariciidae Pavona",
}

# Style B — morphological descriptions (aligns with Veron book text used in ReefNet fine-tune)
genera_descriptive = {
    "Platygyra":    "underwater photo of Platygyra brain coral with maze-like ridges and valleys",
    "Acropora":     "underwater photo of Acropora coral with branching or tabulate form and a single large corallite at each branch tip",
    "Stylophora":   "underwater photo of Stylophora coral with club-shaped lumpy compact branches",
    "Millepora":    "underwater photo of Millepora fire coral with smooth surface and tiny pores forming upright blade-like plates or branching columns without visible corallites",
    "Favites":      "underwater photo of Favites coral with honeycomb pattern of angular corallites with shared walls between adjacent polyps",
    "Dipsastraea": "underwater photo of Dipsastraea coral with rounded separate polyp mounds",
    "Favia": "underwater photo of Favia coral with rounded clearly separated corallites",
    "Echinopora":   "underwater photo of Echinopora coral with thin delicate plating sheets or irregular branching with small projecting corallites",
    "Hydnophora":   "underwater photo of Hydnophora coral with distinctive sharp pointed monticules projecting from surface",
    "Porites":      "underwater photo of Porites coral with very fine small porous uniform texture on massive or branching colony",
    "Acanthastrea": "underwater photo of Acanthastrea coral with large fleshy colorful separate polyps",
    "Cyphastrea":   "underwater photo of Cyphastrea coral with very small corallites embedded in smooth encrusting surface",
    "Goniastrea":   "underwater photo of Goniastrea coral with meandering brain-like valleys or tightly packed rounded corallites on encrusting surface",
    "Seriatopora":  "underwater photo of Seriatopora coral with very thin needle-like delicate branches and pointed tips",
    "Pocillopora":  "underwater photo of Pocillopora coral with cauliflower-like warty compact branches",
    "Montipora":    "underwater photo of Montipora coral with thin smooth encrusting plating sheets or thin finger-like branches with very small corallites",
    "Leptastrea":   "underwater photo of Leptastrea coral with small tightly packed encrusting corallites",
    "Galaxea":      "underwater photo of Galaxea coral with long separate tubular corallites and star-shaped polyps with bright white tips",
    "Lobophyllia":  "underwater photo of Lobophyllia coral with large fleshy lobed polyps",
    "Pavona":       "underwater photo of Pavona coral with thin leaf-like or vertical plates and fine surface texture",
}

# Style C — common name anchored to genus name
# BioCLIP was trained with mixed text types including common names,
# so this provides a third independent embedding direction per genus.
genera_common = {
    "Platygyra":    "brain coral Platygyra",
    "Acropora":     "staghorn coral Acropora",
    "Stylophora":   "club finger coral Stylophora",
    "Millepora":    "fire coral Millepora",
    "Favites":      "moon coral Favites",
    "Dipsastraea":  "pineapple coral Dipsastraea",
    "Favia":        "star coral Favia",
    "Echinopora":   "chalice coral Echinopora",
    "Hydnophora":   "velvet horn coral Hydnophora",
    "Porites":      "boulder coral Porites",
    "Acanthastrea": "button coral Acanthastrea",
    "Cyphastrea":   "encrusting pore coral Cyphastrea",
    "Goniastrea":   "wall brain coral Goniastrea",
    "Seriatopora":  "bird nest coral Seriatopora",
    "Pocillopora":  "cauliflower coral Pocillopora",
    "Montipora":    "velvet plate coral Montipora",
    "Leptastrea":   "dwarf cup coral Leptastrea",
    "Galaxea":      "galaxy coral Galaxea",
    "Lobophyllia":  "lobed brain coral Lobophyllia",
    "Pavona":       "cactus coral Pavona",
}


genera_hybrid = {
    # Update this one line in your hybrid dictionary:
    "Platygyra": "Animalia Cnidaria Anthozoa Scleractinia Merulinidae Platygyra brain coral with maze-like ridges and valleys",
    "Acropora":     "Animalia Cnidaria Anthozoa Scleractinia Acroporidae Acropora",
    "Stylophora":   "Animalia Cnidaria Anthozoa Scleractinia Pocilloporidae Stylophora",
    "Millepora":    "Animalia Cnidaria Hydrozoa Anthoathecata Milleporidae Millepora",
    "Favites":      "Animalia Cnidaria Anthozoa Scleractinia Faviidae Favites",
    "Dipsastraea":  "Animalia Cnidaria Anthozoa Scleractinia Merulinidae Dipsastraea",
    "Favia":        "Animalia Cnidaria Anthozoa Scleractinia Faviidae Favia",
    "Echinopora":   "Animalia Cnidaria Anthozoa Scleractinia Faviidae Echinopora",
    "Hydnophora":   "Animalia Cnidaria Anthozoa Scleractinia Merulinidae Hydnophora",
    "Porites":      "Animalia Cnidaria Anthozoa Scleractinia Poritidae Porites",
}

# Update your master list to pull from the hybrid keys
genera = list(genera_taxonomy.keys())

# ── 3. TEXT ENCODING WITH OPTIONAL ENSEMBLE ──────────────────────────────────
def encode_prompt_list(prompt_list):
    """Encode a list of text prompts → normalised feature matrix (n_classes × dim)."""
    if model_backend == "open_clip":
        tokens = tokenizer(prompt_list).to(device)
        with torch.no_grad():
            feats = model.encode_text(tokens)
    elif model_backend == "transformers":
        inputs = siglip_processor(
            text=prompt_list, return_tensors="pt", padding="max_length"
        ).to(device)
        with torch.no_grad():
            feats = model.get_text_features(**inputs)

            # --- FIX: Extract the tensor from the output object ---
            if not isinstance(feats, torch.Tensor):
                feats = feats.pooler_output

    feats /= feats.norm(dim=-1, keepdim=True)
    return feats

if PROMPT_FORMAT == "taxonomy":
    if model_backend == "transformers":
        print("⚠️  taxonomy format not recommended for SigLIP 2 — switching to descriptive")
        text_prompts  = [genera_descriptive[g] for g in genera]
    else:
        text_prompts  = [genera_taxonomy[g] for g in genera]
    text_features = encode_prompt_list(text_prompts)
    print(f"Using prompt format: taxonomy")
    print(f"Example: {text_prompts[0]}")

elif PROMPT_FORMAT == "descriptive":
    text_prompts  = [genera_descriptive[g] for g in genera]
    text_features = encode_prompt_list(text_prompts)
    print(f"Using prompt format: descriptive")
    print(f"Example: {text_prompts[0]}")

elif PROMPT_FORMAT == "ensemble":
    if model_backend == "transformers":
        # SigLIP 2: skip taxonomy (not trained on Linnaean strings), use 2-way ensemble
        feats_desc = encode_prompt_list([genera_descriptive[g] for g in genera])
        feats_cmn  = encode_prompt_list([genera_common[g]      for g in genera])
        text_features = (feats_desc + feats_cmn) / 2.0
        print("Using prompt format: ensemble (descriptive + common names) — taxonomy skipped for SigLIP 2")
    else:
        feats_tax  = encode_prompt_list([genera_taxonomy[g]    for g in genera])
        feats_desc = encode_prompt_list([genera_descriptive[g] for g in genera])
        feats_cmn  = encode_prompt_list([genera_common[g]      for g in genera])
        text_features = (feats_tax + feats_desc + feats_cmn) / 3.0
        print("Using prompt format: ensemble (taxonomy + descriptive + common names)")
    text_features /= text_features.norm(dim=-1, keepdim=True)
elif PROMPT_FORMAT == "hybrid":
    text_prompts  = [genera_hybrid[g] for g in genera]
    text_features = encode_prompt_list(text_prompts)
    print(f"Using prompt format: hybrid (custom per-genus prompts)")
    print(f"Example Platygyra: {genera_hybrid['Platygyra']}")
    print(f"Example Acropora: {genera_hybrid['Acropora']}")
else:
    raise ValueError(f"Unknown PROMPT_FORMAT: {PROMPT_FORMAT}")

print(f"\n✅ Text features encoded for {len(genera)} genera  |  shape: {text_features.shape}")


# ── 4. CROP FUNCTIONS ───────────────────────────────────────────────


def get_crop(img, bbox, padding_pct=0.15):
    x, y, w, h = map(int, bbox)
    pad_w = int(w * padding_pct)
    pad_h = int(h * padding_pct)
    x1 = max(0, x - pad_w)
    y1 = max(0, y - pad_h)
    x2 = min(img.shape[1], x + w + pad_w)
    y2 = min(img.shape[0], y + h + pad_h)
    crop = img[y1:y2, x1:x2]
    if crop.size == 0:
        return None
    return crop

def get_masked_crop(img, bbox, rle, padding_pct=0.05, background="black", min_visible=500):
    mask = decode_rle(rle)
    if mask.shape != img.shape[:2]:
        mask = cv2.resize(mask.astype(np.uint8),
                          (img.shape[1], img.shape[0]),
                          interpolation=cv2.INTER_NEAREST).astype(bool)
    bg_val = {"black": 0, "gray": 128, "white": 255}.get(background, 0)
    masked_img = img.copy()
    masked_img[~mask] = bg_val
    x, y, w, h = map(int, bbox)
    pad_w = int(w * padding_pct)
    pad_h = int(h * padding_pct)
    x1 = max(0, x - pad_w)
    y1 = max(0, y - pad_h)
    x2 = min(img.shape[1], x + w + pad_w)
    y2 = min(img.shape[0], y + h + pad_h)
    crop = masked_img[y1:y2, x1:x2]
    if crop.size == 0:
        return None
    visible = np.any(crop != bg_val, axis=2).sum()
    if visible < min_visible:
        return None
    return crop

def is_sharp_enough(crop, threshold=50.0):
    gray = cv2.cvtColor(crop, cv2.COLOR_RGB2GRAY)
    return cv2.Laplacian(gray, cv2.CV_64F).var() >= threshold





# ── 5. CONFIGURATION ─────────────────────────────────────────────────────────
BATCH_SIZE = 64
crop_mode  = "masked" if USE_MASK else "bbox"
sample_tag = SAMPLE_MODE if SAMPLE_ONLY else "full"
# Filename encodes all three key settings so runs never overwrite each other
output_csv = os.path.join(
    run_dir,
    f'coral_weak_labels_{MODEL_VERSION}_{PROMPT_FORMAT}_{crop_mode}_{sample_tag}.csv'
)
print(f"\nOutput: {output_csv}")


# ── 6. RESUME LOGIC ──────────────────────────────────────────────────────────
processed_ids = set()
if RESUME and not SAMPLE_ONLY and os.path.exists(output_csv):
    existing      = pd.read_csv(output_csv)
    processed_ids = set(existing['coral_id'])
    print(f"↩️  Resuming: {len(processed_ids)} masks already done, skipping them")


# ── 7. FILE LIST ─────────────────────────────────────────────────────────────

if not SAMPLE_ONLY:
    json_files = json_files_all
    print(f"🚀 Full run: {len(json_files)} files")
else:
    if SAMPLE_MODE == "first":
        json_files = json_files_all[:SAMPLE_SIZE]
    elif SAMPLE_MODE == "random":
        random.seed(RANDOM_SEED)
        json_files = random.sample(json_files_all, min(SAMPLE_SIZE, len(json_files_all)))
    elif SAMPLE_MODE == "stratified":
        from collections import defaultdict
        groups = defaultdict(list)
        for f in json_files_all:
            prefix = f.split('.')[0].rstrip('0123456789')
            groups[prefix].append(f)
        per_group  = max(1, SAMPLE_SIZE // len(groups))
        json_files = []
        random.seed(RANDOM_SEED)
        for prefix, files in sorted(groups.items()):
            json_files.extend(random.sample(files, min(per_group, len(files))))
        json_files = json_files[:SAMPLE_SIZE]
    elif SAMPLE_MODE == "best":
        json_files = df_counts.head(SAMPLE_SIZE)['file'].tolist()
    print(f"🔬 Sample mode: {SAMPLE_MODE} — {len(json_files)} files")


# ── 8. PROCESSING LOOP ───────────────────────────────────────────────────────
metadata_list   = []
scores_list     = []
batch_tensors   = []
batch_meta_temp = []

def process_batch(tensors, metas):
    if not tensors:
        return
    batch_input = torch.stack(tensors).to(device)
    with torch.no_grad():
        if model_backend == "open_clip":
            image_features = model.encode_image(batch_input)
        else:  # transformers / SigLIP 2
            image_features = model.get_image_features(pixel_values=batch_input)
            # --- FIX: Extract the tensor from the output object ---
            if not isinstance(image_features, torch.Tensor):
                image_features = image_features.pooler_output
        image_features /= image_features.norm(dim=-1, keepdim=True)
        # SigLIP 2 uses sigmoid loss so softmax is less meaningful,
        # but softmax over our closed label set still gives comparable confidence scores
        similarity = (100.0 * image_features @ text_features.T).softmax(dim=-1)
    scores_list.append(similarity.cpu().numpy())
    metadata_list.extend(metas)



total_before_filter = 0
total_after_filter  = 0

for j_file in tqdm(json_files, desc="BioCLIP weak labeling"):
    path = os.path.join(json_dir, j_file)
    with open(path, 'r') as f:
        data = json.load(f)

    img_name = data['image']['file_name']
    img_path = os.path.join(data_path, img_name)
    if not os.path.exists(img_path):
        continue

    image = cv2.imread(img_path)
    if image is None:
        continue
    image = cv2.resize(image, (2560, 1920))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    total_before_filter += len(data['annotations'])

    filtered_annos = filter_annotations(
        data['annotations'],
        min_area=MIN_AREA,
        containment_thresh=CONTAINMENT_THRESH,
        max_aspect_ratio=MAX_ASPECT_RATIO,
    )
    total_after_filter += len(filtered_annos)

    for anno in filtered_annos:
        if anno['id'] in processed_ids:
            continue

        if USE_MASK:
            crop_np = get_masked_crop(
                image, anno['bbox'], anno['segmentation'],
                padding_pct=MASK_PADDING_PCT,
                background=MASK_BACKGROUND,
            )
        else:
            crop_np = get_crop(image, anno['bbox'])

        if crop_np is None:
            continue
        if not is_sharp_enough(crop_np, threshold=SHARPNESS_THRESHOLD):
            continue

        crop_pil = Image.fromarray(crop_np)
        if model_backend == "open_clip":
            img_tensor = preprocess_val(crop_pil)
        else:  # transformers / SigLIP 2
            img_tensor = siglip_processor(
                images=crop_pil, return_tensors="pt"
            )['pixel_values'].squeeze(0)



        batch_tensors.append(img_tensor)
        batch_meta_temp.append({
            'image_name': img_name,
            'coral_id':   anno['id'],
            'bbox':       anno['bbox'],
            'area':       anno['area'],
        })

        if len(batch_tensors) >= BATCH_SIZE:
            process_batch(batch_tensors, batch_meta_temp)
            batch_tensors   = []
            batch_meta_temp = []

if batch_tensors:
    process_batch(batch_tensors, batch_meta_temp)

print(f"\nAnnotations before filtering: {total_before_filter}")
print(f"Annotations after filtering:  {total_after_filter} "
      f"({100*total_after_filter/max(total_before_filter,1):.1f}% kept)")


# ── 9. SAVE RESULTS ──────────────────────────────────────────────────────────
if not metadata_list:
    print("⚠️ Nothing new to save.")
else:
    df_meta    = pd.DataFrame(metadata_list)
    all_scores = np.vstack(scores_list)

    df_scores = pd.DataFrame(all_scores, columns=genera)
    df_scores['predicted_genus'] = df_scores.idxmax(axis=1)
    df_scores['confidence']      = df_scores[genera].max(axis=1)

    final_df = pd.concat([df_meta, df_scores], axis=1)

    # >>> ADAPTED (3): the legacy code wrote final_df to output_csv here; it stays in memory now
    print(f"✅ {len(final_df)} corals in final_df (in memory, nothing written to Drive)")
    print(f"\nPredicted genera distribution:")
    print(final_df['predicted_genus'].value_counts())
    print(f"\nMean confidence: {final_df['confidence'].mean():.3f}")
    print(f"High-confidence (≥0.50): {(final_df['confidence']>=0.5).sum()} / {len(final_df)}")

# >>> ADAPTED (4): legacy section 10 (confidence histogram) is omitted; it saved a PNG into run_dir

In [ ]:
import gc

from coralid.encoders import load_encoder
from coralid.evaluation import compare_predictions, load_legacy_predictions
from coralid.genera import load_genera
from coralid.manifest import image_manifest
from coralid.masks import load_mask_json
from coralid.orientation import PORTRAIT, image_orientations
from coralid.parity import (
    check_against_drive_csv,
    check_sample_counts,
    check_zero_shot_parity,
    compare_by_orientation,
    counts_by_orientation,
    manifest_counts,
)
from coralid.zero_shot import build_text_features, run_zero_shot

# --- keep what the legacy code produced, then free its model --------------------------------
legacy_reference = final_df.copy()
legacy_counts = {
    "masks": total_before_filter,
    "after_filter": total_after_filter,
    "crops": len(final_df),
}
sample_json_files = list(json_files)  # the 50 files the legacy code used, in its order
for name in ("model", "text_features", "tokenizer", "preprocess_val", "siglip_processor"):
    globals().pop(name, None)
gc.collect()
torch.cuda.empty_cache()
print("legacy reference:", legacy_counts)

# --- the same configuration through coralid --------------------------------------------------
sample_images = [load_mask_json(Path(cfg.json_dir) / f).image_name for f in sample_json_files]
t0 = time.perf_counter()
sample_manifest = image_manifest(
    cfg.json_dir, cfg.images_dir, cfg.filters, cfg.crops, image_names=sample_images, progress=True
)
print(
    f"manifest of the sample: {time.perf_counter() - t0:.0f} s, {int(sample_manifest['keep'].sum())} masks kept"
)

zs = cfg.zero_shot
encoder = load_encoder(zs.model)
genera = load_genera().genera(zs.label_set)
text_feats = build_text_features(encoder, zs.label_set, zs.prompt_format, zs.prompt_set)
t0 = time.perf_counter()
result = run_zero_shot(
    encoder,
    text_feats,
    genera,
    sample_manifest,
    cfg.json_dir,
    cfg.images_dir,
    crops=cfg.crops,
    batch_size=zs.batch_size,
    logit_scale=zs.logit_scale,
    progress=True,
)
new_predictions = result.predictions
print(f"coralid zero-shot: {time.perf_counter() - t0:.0f} s, {len(new_predictions)} predictions")
if result.failures:
    print("FAILURES (these images were not classified):", result.failures)

# --- compare, landscape photos and portrait photos apart (a = legacy, b = coralid) ------------
frames = sample_manifest[["image_name", "frame_w", "frame_h"]].drop_duplicates()
orientations = image_orientations(frames)
print("photos in the sample:", orientations.value_counts().to_dict())
comparisons = compare_by_orientation(legacy_reference, new_predictions, frames)
for orientation, comparison in comparisons.items():
    print(f"\n--- {orientation} photos ---")
    print(comparison)
    if len(comparison.disagreements):
        print("top-1 disagreements (first 20):")
        print(comparison.disagreements.to_string())

rows_b = check_sample_counts(
    legacy_counts,
    manifest_counts(sample_manifest),
    counts_by_orientation(legacy_reference, frames),
    counts_by_orientation(new_predictions, frames),
)
rows_b += check_zero_shot_parity(
    comparisons, portrait_images=int((orientations == PORTRAIT).sum())
)

# --- informational: the file saved on Drive --------------------------------------------------
drive_csv = Path(cfg.run_dir) / "coral_weak_labels_finetuned-bioclip_taxonomy_masked_best.csv"
if drive_csv.exists():
    drive_comparison = compare_predictions(new_predictions, load_legacy_predictions(drive_csv))
    rows_b += check_against_drive_csv(drive_comparison)
else:
    print("(informational) not found:", drive_csv)

ROWS["B"] = rows_b
print()
print(format_table(rows_b))

## Check D: the expert test set (seconds)

Evaluates the predictions stored *in* `assaf_expert_test_set.csv` against that file's own expert labels with `coralid.evaluation`, and compares the result cell by cell with the confusion matrix of the real file: **77 rows**, 54 in the 20-genus label set, 5 soft corals and 18 with no genus, 36 correct (36/54 = 66.7%). The 18 rows without a genus have an empty `expert_genus`; their status comes from `is_correct` (`unidentifiable` or `MISSING_LABEL`). The memory matrix of 78 rows (39/57) is a slightly different version of the same review and is no longer the reference. It prints the labels found in the file first, so a surprise in the label wording is visible. Reads one small CSV.

In [ ]:
from coralid.evaluation import evaluate, load_expert_set, load_legacy_predictions
from coralid.parity_expert import check_expert_confusion

EXPERT_CSV = cfg.run_dir / "assaf_expert_test_set.csv"

predictions = load_legacy_predictions(EXPERT_CSV)
expert = load_expert_set(EXPERT_CSV, "hard20")
print(f"{len(expert)} expert rows. Labels found in the file (empty expert_genus read from is_correct):")
print(expert["expert_raw"].value_counts(dropna=False).to_string())

report = evaluate(predictions, expert, "hard20")
print("\ncoverage:", report.coverage)
print("rows per status:", report.status_counts)
for title, acc in (
    ("in the label set", report.accuracy_in_set),
    ("in the label set + soft corals", report.accuracy_in_plus_out),
):
    low, high = acc["ci95"] or (float("nan"), float("nan"))
    print(
        f"accuracy, {title}: {acc['correct']}/{acc['total']} = {acc['accuracy']:.3f}  (Wilson 95% {low:.3f}-{high:.3f})"
    )
print("\nper genus (recall = correct / true count, precision = correct / predicted count):")
print(report.per_genus.round(3).to_string())
print("\ntop confusions:")
print(report.top_confusions.to_string(index=False))

ROWS["D"] = check_expert_confusion(report)  # expectations of the real file: coralid.parity_expert
print("\nagainst the real file (one line per confusion cell):")
print(format_table(ROWS["D"], show_cells=True))

## Check C: image manifest on all images (resumable)

Runs `image_manifest` over every JSON and photo: the geometric filters, then visibility and sharpness of each masked crop. Expect 44,211 masks. **If `<run>/manifest/manifest.parquet` is already saved (the first run saved it), it is loaded and left alone and this cell takes seconds.** Otherwise progress is saved to `<run>/manifest/parts/` every 25 images, so you can stop the cell (or lose the connection) and run it again: finished images are skipped and failed images are retried. Stage B builds on this file.

The check is about completeness: all 44,211 masks are in the manifest and no photo failed. The kept counts (per orientation, the total against the recorded 16,080, the rejection reasons) are reported as information. The legacy total of 14,224 is no longer a target, because the legacy code cut the wrong region on portrait photos; the first run kept 16,080.

In [ ]:
from coralid.batch import ResumableRun
from coralid.manifest import image_manifest, read_manifest, write_manifest
from coralid.orientation import by_orientation
from coralid.parity_manifest import check_image_manifest

manifest_path = cfg.manifest_path  # <run>/manifest/manifest.parquet
if manifest_path.exists():
    print(
        "The manifest is already saved; loading it. Nothing is recomputed or overwritten:",
        manifest_path,
    )
    full_manifest = read_manifest(manifest_path)
    unresolved = []
else:
    run_c = ResumableRun(manifest_path.parent / "parts", "manifest", flush_every=25)
    print(
        f"{len(run_c.done)} images already done from an earlier attempt; continuing with the rest"
    )
    t0 = time.perf_counter()
    full_manifest = image_manifest(
        cfg.json_dir, cfg.images_dir, cfg.filters, cfg.crops, progress=True, run=run_c
    )
    print(f"{time.perf_counter() - t0:.0f} s this time")
    unresolved = run_c.unresolved_failures()
    if unresolved:
        print(
            f"{len(unresolved)} image(s) failed and were NOT saved as done. Run this cell again to retry them:"
        )
        for failure in unresolved[:10]:
            print("  ", failure["image"], failure["error"])
    else:
        saved = write_manifest(
            full_manifest, manifest_path, config=cfg, overwrite=False, run_name=cfg.run_name
        )
        print("saved (new files):", saved, "and", saved.with_suffix(".meta.json"))

per_orientation = by_orientation(
    full_manifest,
    full_manifest,
    lambda df: {"masks": len(df), "kept": int(df["keep"].sum()), "photos": df["image_name"].nunique()},
)
print("masks, kept masks and photos per orientation:", per_orientation)

ROWS["C"] = check_image_manifest(full_manifest, unresolved)
print()
print(format_table(ROWS["C"]))

In [ ]:
all_rows = [row for letter in "ABCD" for row in ROWS.get(letter, [])]
summary = summarize(all_rows)

print(format_summary(summary))
print()
print(format_table([row for row in all_rows if row.check != "D"]))

report_path = write_report(
    summary, cfg.reports_dir, extra={"run_name": cfg.run_name, "branch": BRANCH, "commit": commit}
)
print("\nSaved (a new file, nothing was overwritten):", report_path)
print("\n" + "=" * 30 + " PASTE THIS BACK " + "=" * 30)
print(paste_block(summary, {"branch": BRANCH, "commit": commit}))